## WORK IN PROGRESS
IDEA IS TO PROVIDE HELP TO FIND MOST RELEVANT ENTITIES <br>
OR <br>
IT MAY PROVIDE CONCRETE IDEAS ON WHAT OR NOT WHAT TO DO

In [1]:
# 0. IMPORTS
import json
import requests
import pandas as pd
import duckdb
import re

### Using the Term Search https://data.bioontology.org/documentation#nav_search from NCBO API to find possible relevant concepts in ontology 

<img src="Term_Search.png" alt="Term Search section from https://data.bioontology.org/documentation#nav_search" height="500">

In [2]:
# 1. Configuration + User defined parameter
API_KEY = "INSERT_YOUR_API_KEY_HERE"
SEARCH_URL = "https://data.bioontology.org/search"
query_text = input("Please enter query text: ")

Please enter query text:  waste water


In [3]:
# 2. Define the payload parameters
payload = {
    "q": query_text,    
    # "ontologies": "NCIT,MESH,DOID" # Optional: Limit evaluation to specific ontologies
}

In [4]:
# 3. Setup authentication header
headers = {
    "Authorization": f"apikey token={API_KEY}",
    "Content-Type": "application/json"
}

In [5]:
# 4. Fire the API call
print("Querying the BioPortal Search API...")
response = requests.post(SEARCH_URL, json=payload, headers=headers)
print(f"Response Status Code is {response}")

Querying the BioPortal Search API...
Response Status Code is <Response [200]>


In [28]:
# 5. Process and format the results

# Column headers for the pandas DataFrame
row_header = [
    "searchTerm", "prefLabel", "synonym", "other synonyms", "definition", "cui",
    "semanticType", "ontologyName", "ontologyLink", "ontologyId", "url"
]

entity_list = []

if response.status_code == 200:
    results = response.json()

    entity_list = [
        (
            query_text,
            e["prefLabel"],
            (e.get("synonym") or [None])[0],
            (e.get("synonym") or [])[1:] or None,
            (e.get("definition") or [None])[0],
            e.get("cui"),
            e.get("semanticType"),
            e["links"]["ontology"].rsplit("/", 1)[-1],
            e["links"]["ontology"],
            e["@id"].rsplit("/", 1)[-1],
            e["@id"],
        )
        for e in results.get("collection", [])
    ]

else:
    print(f"Failed to fetch recommendations. Status code: {response.status_code}")

In [29]:
df = pd.DataFrame(
 entity_list,
 columns=row_header)

In [30]:
display(df.head(50))

,searchTerm,prefLabel,synonym,other synonyms,definition,cui,semanticType,ontologyName,ontologyLink,ontologyId,url
0,waste water,Waste water,Waste water (substance),None,None,[C0450237],[T167],SNOMEDCT,https://data.bioontology.org/ontologies/SNOMEDCT,264327008,http://purl.bioontology.org/ontology/SNOMEDCT/264327008
1,waste water,Waste water,None,None,None,[C0450237],[T167],RCD,https://data.bioontology.org/ontologies/RCD,XC0RW,http://purl.bioontology.org/ontology/RCD/XC0RW
2,waste water,Status of waste water systems,None,None,None,[C4738123],[T033],LOINC,https://data.bioontology.org/ontologies/LOINC,LP269986-8,http://purl.bioontology.org/ontology/LNC/LP269986-8
3,waste water,Status of waste water systems,None,None,None,[C4738123],[T033],LOINC,https://data.bioontology.org/ontologies/LOINC,MTHU059023,http://purl.bioontology.org/ontology/LNC/MTHU059023
4,waste water,waste water,wastewater,None,Water that has been adversely affected in quality by anthropogenic influence.,None,None,MEO,https://data.bioontology.org/ontologies/MEO,MEO_0000035,https://mdatahub.org/data/meo/MEO_0000035
5,waste water,waste water,None,None,None,None,None,OCHV,https://data.bioontology.org/ontologies/OCHV,ochv#C0450237,http://sbmi.uth.tmc.edu/ontology/ochv#C0450237
6,waste water,Waste Water,None,None,None,None,None,RH-MESH,https://data.bioontology.org/ontologies/RH-MESH,mesh.owl#D062065,http://phenomebrowser.net/ontologies/mesh/mesh.owl#D062065
7,waste water,Waste Water,None,None,None,None,None,RH-MESH,https://data.bioontology.org/ontologies/RH-MESH,mesh.owl#D20.944.932,http://phenomebrowser.net/ontologies/mesh/mesh.owl#D20.944.932
8,waste water,Waste Water,None,None,None,None,None,RH-MESH,https://data.bioontology.org/ontologies/RH-MESH,mesh.owl#N06.850.460.710.865,http://phenomebrowser.net/ontologies/mesh/mesh.owl#N06.850.460.710.865
9,waste water,waste water,None,None,Water that has been adversely affected in quality by anthropogenic influence.,None,None,BERO,https://data.bioontology.org/ontologies/BERO,ENVO_00002001,http://purl.obolibrary.org/obo/ENVO_00002001


In [16]:
pd.set_option('display.max_colwidth', 200)

In [32]:
#Pandas way
df[['searchTerm', 'definition', 'ontologyName', 'ontologyId', 'url']]

,searchTerm,definition,ontologyName,ontologyId,url
0,waste water,None,SNOMEDCT,264327008,http://purl.bioontology.org/ontology/SNOMEDCT/264327008
1,waste water,None,RCD,XC0RW,http://purl.bioontology.org/ontology/RCD/XC0RW
2,waste water,None,LOINC,LP269986-8,http://purl.bioontology.org/ontology/LNC/LP269986-8
3,waste water,None,LOINC,MTHU059023,http://purl.bioontology.org/ontology/LNC/MTHU059023
4,waste water,Water that has been adversely affected in quality by anthropogenic influence.,MEO,MEO_0000035,https://mdatahub.org/data/meo/MEO_0000035
5,waste water,None,OCHV,ochv#C0450237,http://sbmi.uth.tmc.edu/ontology/ochv#C0450237
6,waste water,None,RH-MESH,mesh.owl#D062065,http://phenomebrowser.net/ontologies/mesh/mesh.owl#D062065
7,waste water,None,RH-MESH,mesh.owl#D20.944.932,http://phenomebrowser.net/ontologies/mesh/mesh.owl#D20.944.932
8,waste water,None,RH-MESH,mesh.owl#N06.850.460.710.865,http://phenomebrowser.net/ontologies/mesh/mesh.owl#N06.850.460.710.865
9,waste water,Water that has been adversely affected in quality by anthropogenic influence.,BERO,ENVO_00002001,http://purl.obolibrary.org/obo/ENVO_00002001


In [33]:
#DuckDB way

# Query the DataFrames directly using their variable names
query = """
    SELECT searchTerm, definition, ontologyName, ontologyId, url
    FROM df
"""

# Execute and return the result as a new Pandas DataFrame
result_df = duckdb.sql(query).df()
display(result_df)

,searchTerm,definition,ontologyName,ontologyId,url
0,waste water,None,SNOMEDCT,264327008,http://purl.bioontology.org/ontology/SNOMEDCT/264327008
1,waste water,None,RCD,XC0RW,http://purl.bioontology.org/ontology/RCD/XC0RW
2,waste water,None,LOINC,LP269986-8,http://purl.bioontology.org/ontology/LNC/LP269986-8
3,waste water,None,LOINC,MTHU059023,http://purl.bioontology.org/ontology/LNC/MTHU059023
4,waste water,Water that has been adversely affected in quality by anthropogenic influence.,MEO,MEO_0000035,https://mdatahub.org/data/meo/MEO_0000035
5,waste water,None,OCHV,ochv#C0450237,http://sbmi.uth.tmc.edu/ontology/ochv#C0450237
6,waste water,None,RH-MESH,mesh.owl#D062065,http://phenomebrowser.net/ontologies/mesh/mesh.owl#D062065
7,waste water,None,RH-MESH,mesh.owl#D20.944.932,http://phenomebrowser.net/ontologies/mesh/mesh.owl#D20.944.932
8,waste water,None,RH-MESH,mesh.owl#N06.850.460.710.865,http://phenomebrowser.net/ontologies/mesh/mesh.owl#N06.850.460.710.865
9,waste water,Water that has been adversely affected in quality by anthropogenic influence.,BERO,ENVO_00002001,http://purl.obolibrary.org/obo/ENVO_00002001


In [41]:
# DuckDB way to collect how many times an entity is listed from Term Search (This is not even from crawling from 

# Query the DataFrames directly using their variable names
query = """
    SELECT url, COUNT(ontologyId) AS idCount
    FROM df
    GROUP BY url
    ORDER BY idCount DESC
"""

# Execute and return the result as a new Pandas DataFrame
result_df = duckdb.sql(query).df()
display(result_df)

,url,idCount
0,http://purl.obolibrary.org/obo/ENVO_00002001,13
1,http://purl.obolibrary.org/obo/OMIT_0028035,1
2,http://purl.bioontology.org/ontology/MESH/D008506,1
3,http://purl.bioontology.org/ontology/MESH/D059029,1
4,http://purl.jp/bio/4/id/200906082976266130,1
5,http://purl.bioontology.org/ontology/SNOMEDCT/1290096006,1
6,http://purl.bioontology.org/ontology/RCD/XC0RW,1
7,http://purl.obolibrary.org/obo/MESH_D062065,1
8,http://purl.jp/bio/4/id/201006057435647250,1
9,http://purl.bioontology.org/ontology/SNOMEDCT/289925000,1


In [67]:
FILENAME = "C:/Users/Clari/Downloads/Entities_from_CQs_WORKING_OUT.tsv"
FILENAME = "../Entities_from_CQs_WORKING_OUT.tsv"

In [68]:
entity_working_df = pd.read_csv(FILENAME, sep='\t', usecols=[3,4,5,6])
#entity_working_df = pd.read_csv(FILENAME, sep='\t')

In [69]:
display(entity_working_df)

,From CQ 10 (soil microbiome),Refined Entities Notes.1,From CQ 11 (soil microbiome),Unnamed: 6
0,"Proteobacteria, Actinobacteria, Acidobacteria, Chloroflexi, Planctomycetes, Rhizobium, Azotobacter, Azospirillum, Clostridium, Taxonomic composition, Shannon diversity index, Simpson diversity ind...",NaN,"Proteobacteria, Actinobacteria, Acidobacteria, Chloroflexi, Planctomycetes, Rhizobium, Azotobacter, Azospirillum, Clostridium, Taxonomic composition, Shannon diversity index, Simpson diversity ind...",NaN
1,Acidobacteria,taxon: include,Acidobacteria,taxon: include
2,Acidophile,taxon: include,Acidophile,taxon: include
3,Actinobacteria,taxon: include,Actinobacteria,taxon: include
4,arsenic,chemical: include?,Azospirillum,taxon: include
...,...,...,...,...
67,NaN,NaN,NaN,NaN
68,NaN,NaN,NaN,NaN
69,NaN,NaN,NaN,NaN
70,NaN,NaN,NaN,NaN


In [71]:
c_entity_working_df = entity_working_df.dropna(how='all')
display(c_entity_working_df)

,From CQ 10 (soil microbiome),Refined Entities Notes.1,From CQ 11 (soil microbiome),Unnamed: 6
0,"Proteobacteria, Actinobacteria, Acidobacteria, Chloroflexi, Planctomycetes, Rhizobium, Azotobacter, Azospirillum, Clostridium, Taxonomic composition, Shannon diversity index, Simpson diversity ind...",NaN,"Proteobacteria, Actinobacteria, Acidobacteria, Chloroflexi, Planctomycetes, Rhizobium, Azotobacter, Azospirillum, Clostridium, Taxonomic composition, Shannon diversity index, Simpson diversity ind...",NaN
1,Acidobacteria,taxon: include,Acidobacteria,taxon: include
2,Acidophile,taxon: include,Acidophile,taxon: include
3,Actinobacteria,taxon: include,Actinobacteria,taxon: include
4,arsenic,chemical: include?,Azospirillum,taxon: include
5,Azospirillum,taxon: include,Azotobacter,taxon: include
6,Azotobacter,taxon: include,Biomarker,NaN
7,cadmium,taxon: include,Carbon cycling,?
8,Carbon cycling,?,Chemotroph,include as chemotroph
9,Chloroflexi,?,Chloroflexi,NaN


In [81]:
cewdf = c_entity_working_df.iloc[:,0:2]

In [ ]:
df = df[~df['col'].isin(['val1', 'val2'])]

In [82]:
df = cewdf[~cewdf.iloc[:, 1].str.contains('do not include', na=False)]

In [88]:
df = cewdf[~cewdf.iloc[:, 1].isin(['do not include', 'taxon: include'])]

In [99]:
#Extraction code using regex was generated by ChatGPT

import re
targets = ['do not include', 'taxon: include']

pattern = r'(?<!\w)(?:' + '|'.join(re.escape(t) for t in targets) + r')(?!\w)'

fil_cewdf = cewdf[
    ~cewdf.iloc[:, 1].astype(str).str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )
]

In [100]:
display(fil_cewdf)

,From CQ 10 (soil microbiome),Refined Entities Notes.1
0,"Proteobacteria, Actinobacteria, Acidobacteria, Chloroflexi, Planctomycetes, Rhizobium, Azotobacter, Azospirillum, Clostridium, Taxonomic composition, Shannon diversity index, Simpson diversity ind...",NaN
4,arsenic,chemical: include?
8,Carbon cycling,?
9,Chloroflexi,?
10,chromium,include as chemical
11,Clay,soilType
13,CO₂ fixation pathway,include as pathway
14,copper,include as chemical?
15,Halophile,include as halophile
17,Lead,include as chemical?


In [102]:
fil_cewdf.iloc[:,2] = (
    fil_cewdf.iloc[:, 1]
    .astype(str)
    .str.replace(r'^include as ', '', regex=True, case=False)
    .str.replace(r'\?$', '', regex=True)
    .str.strip()
)

In [103]:
display(fil_cewdf)

,From CQ 10 (soil microbiome),Refined Entities Notes.1,new_column
0,"Proteobacteria, Actinobacteria, Acidobacteria, Chloroflexi, Planctomycetes, Rhizobium, Azotobacter, Azospirillum, Clostridium, Taxonomic composition, Shannon diversity index, Simpson diversity ind...",NaN,nan
4,arsenic,chemical: include?,chemical: include
8,Carbon cycling,?,
9,Chloroflexi,?,
10,chromium,include as chemical,chemical
11,Clay,soilType,soilType
13,CO₂ fixation pathway,include as pathway,pathway
14,copper,include as chemical?,chemical
15,Halophile,include as halophile,halophile
17,Lead,include as chemical?,chemical


### Need to implement finding relating mappings using Mapping API

<img src="Mapping.png" alt="Mapping section from https://data.bioontology.org/documentation#Mapping" height="500">